# Create American Ornithological Society Research Awards

Creates OpenAlex award rows from official American Ornithological Society (AOS) Kessel Fellowship and Latin American/Caribbean Conservation Research Grant announcement pages.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/aos_research_awards_to_s3.py` to download the official AOS pages, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes all parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data sources:** official AOS sitemap plus Kessel/LACCR announcement pages under `https://americanornithology.org/`  
**S3 location:** `s3a://openalex-ingest/awards/aos/aos_research_awards.parquet`  
**Local full-source validation on 2026-05-28:** 35 rows, 2020-2026; 19 Kessel Fellowship rows with exact USD amount coverage; 16 LACCR Grant rows with amount/currency NULL because exact per-recipient amounts are not published.

**American Ornithological Society funder:**
- funder_id: 4320313553
- display_name: "American Ornithological Society"
- country: US

**Funder ambiguity note:** OpenAlex also has legacy rows for American Ornithologists' Union (F4320309061) and Cooper Ornithological Society (F4320314773). These post-2016 award announcements are issued by the current American Ornithological Society, so this ingest maps to F4320313553.

**Amount/currency decision:** Kessel Fellowship rows use exact USD amounts stated in AOS announcements (mostly USD 15,000; one 2025 Arctic fellowship at USD 30,000). LACCR rows keep amount/currency NULL by source authority because official pages publish only an up-to-USD-5,000 program cap, not exact recipient-level amounts; the cap is preserved in raw `program_amount_text` and `amount_note`.

**Mapping summary:** One OpenAlex award row per listed recipient. Native key is `aos-{scheme}-{year}-{recipient_slug}-{source_hash}`. Recipients map to `lead_investigator`, with source affiliation/role mapped to `lead_investigator.affiliation.name`.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aos_research_awards_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/aos/aos_research_awards.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-28 found 35 rows.
SELECT COUNT(*) as total_aos_research_awards
FROM openalex.awards.aos_research_awards_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.aos_research_awards_raw;


In [ ]:
%sql
-- Sample raw AOS data.
SELECT
    funder_award_id,
    recipient_name,
    affiliation,
    project_title,
    award_year,
    funder_scheme,
    amount,
    currency,
    program_amount_text,
    landing_page_url
FROM openalex.awards.aos_research_awards_raw
ORDER BY TRY_CAST(award_year AS INT), funder_scheme, recipient_name
LIMIT 35;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys


In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'aos_research_awards_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|obligated|outlay|grant|award|donation|support|stipend|prize';


In [ ]:
%sql
-- Confirm coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(funder_award_id) AS rows_with_native_id,
    COUNT(recipient_name) AS rows_with_recipient_name,
    COUNT(given_name) AS rows_with_given_name,
    COUNT(family_name) AS rows_with_family_name,
    COUNT(affiliation) AS rows_with_affiliation,
    COUNT(project_title) AS rows_with_project_title,
    COUNT(amount) AS rows_with_exact_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_exact_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_exact_amount_usd,
    MIN(TRY_CAST(award_year AS INT)) AS min_award_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_award_year,
    collect_set(currency) AS exact_amount_currencies
FROM openalex.awards.aos_research_awards_raw;


In [ ]:
%sql
-- Native-key inspection. funder_award_id must be unique.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT source_hash) AS distinct_source_hashes,
    COUNT(DISTINCT recipient_name) AS distinct_recipients
FROM openalex.awards.aos_research_awards_raw;


In [ ]:
%sql
-- Scheme and amount coverage distribution.
SELECT
    funder_scheme,
    funding_type,
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_exact_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_exact_amount,
    ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS total_exact_amount_usd,
    collect_set(program_amount_text) AS program_amount_texts
FROM openalex.awards.aos_research_awards_raw
GROUP BY funder_scheme, funding_type
ORDER BY rows DESC;


In [ ]:
%sql
-- Confirm LACCR exact amounts are intentionally NULL and documented.
SELECT recipient_name, award_year, amount, currency, program_amount_text, amount_note
FROM openalex.awards.aos_research_awards_raw
WHERE funder_scheme = 'Latin American/Caribbean Conservation Research Grant'
ORDER BY TRY_CAST(award_year AS INT), recipient_name;


## Step 1.6: Funder Existence Fail-Fast


In [ ]:
%sql
-- Must return TRUE. If this fails, STOP: American Ornithological Society F4320313553 is missing from openalex.common.funder.
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for American Ornithological Society F4320313553'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320313553;


In [ ]:
%sql
-- Inspect the funder row used for the award struct.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320313553;


## Step 2: Transform to OpenAlex Award Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aos_research_awards
USING delta
AS
WITH
aos_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320313553
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        NULLIF(TRIM(currency), '') AS parsed_currency,
        TRY_TO_DATE(CONCAT(award_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(CONCAT(award_year, '-12-31'), 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(award_year AS INT) AS parsed_award_year
    FROM openalex.awards.aos_research_awards_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
      AND recipient_name IS NOT NULL
      AND TRIM(recipient_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        g.parsed_amount as amount,
        g.parsed_currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        NULLIF(TRIM(g.funding_type), '') as funding_type,
        NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,

        'aos_kessel_laccr_research' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        g.parsed_award_year as start_year,
        g.parsed_award_year as end_year,

        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.affiliation), '') as name,
                CAST(NULL AS STRING) as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN aos_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 180


In [ ]:
%sql
-- Remove previous AOS Kessel/LACCR research award data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aos_kessel_laccr_research' AND priority = 180;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    180 as priority
FROM openalex.awards.aos_research_awards;


## Step 6: Verification Queries


In [ ]:
%sql
SELECT COUNT(*) as total_aos_research_awards
FROM openalex.awards.aos_research_awards;


In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.aos_research_awards;


In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_year,
    lead_investigator.given_name AS given_name,
    lead_investigator.family_name AS family_name,
    lead_investigator.affiliation.name AS source_affiliation,
    landing_page_url
FROM openalex.awards.aos_research_awards
ORDER BY start_year, funder_scheme, display_name;


In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.aos_research_awards;


In [ ]:
%sql
-- Check funder distribution. Should be only American Ornithological Society.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.aos_research_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;


In [ ]:
%sql
-- Data completeness and exact amount coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_exact_amount,
    COUNT(currency) as has_currency,
    COUNT(start_date) as has_start_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.given_name) as has_given_name,
    COUNT(lead_investigator.family_name) as has_family_name,
    COUNT(lead_investigator.affiliation.name) as has_source_affiliation,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_exact_amount,
    ROUND(SUM(amount), 0) as total_exact_usd
FROM openalex.awards.aos_research_awards;


In [ ]:
%sql
-- Per-scheme exact amount coverage; LACCR should remain NULL by source authority.
SELECT
    funder_scheme,
    funding_type,
    COUNT(*) AS rows,
    COUNT(amount) AS rows_with_exact_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_exact_amount,
    ROUND(SUM(amount), 0) AS total_exact_usd,
    collect_set(currency) AS currencies
FROM openalex.awards.aos_research_awards
GROUP BY funder_scheme, funding_type
ORDER BY rows DESC;


In [ ]:
%sql
-- Year distribution.
SELECT start_year, funder_scheme, COUNT(*) AS cnt, ROUND(SUM(amount), 0) AS total_exact_usd
FROM openalex.awards.aos_research_awards
GROUP BY start_year, funder_scheme
ORDER BY start_year, funder_scheme;


In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 180.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids,
    ROUND(SUM(amount), 0) as total_exact_usd
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aos_kessel_laccr_research' AND priority = 180
GROUP BY priority, provenance;


## Handoff / Admin Notes

Contractor-complete handoff only. The contractor has no S3 or Databricks access, so an admin must:

1. Run `scripts/local/aos_research_awards_to_s3.py` to upload `s3://openalex-ingest/awards/aos/aos_research_awards.parquet`.
2. Run this notebook on Databricks.
3. Run the Step 6 verification cells and QA the inserted `provenance='aos_kessel_laccr_research'`, `priority=180` rows.
4. Mark the tracker row Complete after production API verification.

Do not add job YAML until an admin has run and verified the ingest.
